# ANTIDOTE: Additional Comparison and Few-Shot Figures

This notebook loads the merged results in `ALL_RESULTS.json`,

generates additional summary plots (overall method comparison and few-shot

benefit), and saves the resulting figures into the existing `figures/`

directory so they can be used in the report or paper.

In [1]:
# 1. Configure environment and paths

import os

from pathlib import Path



# Base paths: mirror the /kaggle/working layout but keep relative so

# the notebook also runs in the repo root.

BASE_DIR = Path(".").resolve()

RESULTS_DIR = BASE_DIR  # ALL_RESULTS.json lives in the working directory

FIGURES_DIR = BASE_DIR / "figures"



FIGURES_DIR.mkdir(parents=True, exist_ok=True)



ALL_RESULTS_PATH = RESULTS_DIR / "ALL_RESULTS.json"

print(f"Using results file: {ALL_RESULTS_PATH}")

print(f"Figures will be saved to: {FIGURES_DIR}")

Using results file: /Users/oluwadabiraomotoso/Desktop/CMPE 401/antidote/ALL_RESULTS.json
Figures will be saved to: /Users/oluwadabiraomotoso/Desktop/CMPE 401/antidote/figures


In [2]:
# 2. Imports and plotting style

import json

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns



# If utils.py defines any shared styles or constants, import them here.

try:

    import utils  # noqa: F401

except ImportError:

    utils = None



plt.style.use("seaborn-v0_8")

sns.set_context("talk")

sns.set_style("whitegrid")



print("Imports complete; matplotlib/seaborn styles configured.")

Imports complete; matplotlib/seaborn styles configured.


In [3]:
# 3. Load ALL_RESULTS.json

assert ALL_RESULTS_PATH.exists(), f"Missing results file: {ALL_RESULTS_PATH}"



with ALL_RESULTS_PATH.open("r") as f:

    all_results = json.load(f)



# Basic sanity checks on keys and metric ranges

expected_methods = [

    "static_joint",

    "naive_sequential",

    "ewc_only",

    "replay_only",

    "ewc_plus_replay",

]



for m in expected_methods:

    assert m in all_results, f"Expected method '{m}' not found in ALL_RESULTS.json"

    avg_f1 = all_results[m]["avg_f1"]

    avg_acc = all_results[m]["cl_metrics"]["AvgAcc"]

    assert 0.0 <= avg_f1 <= 1.0, f"avg_f1 out of range for {m}: {avg_f1}"

    assert 0.0 <= avg_acc <= 1.0, f"AvgAcc out of range for {m}: {avg_acc}"



assert "few_shot" in all_results, "Missing 'few_shot' section in ALL_RESULTS.json"



print("Loaded ALL_RESULTS.json with methods:", list(all_results.keys()))

Loaded ALL_RESULTS.json with methods: ['static_joint', 'naive_sequential', 'ewc_only', 'replay_only', 'ewc_plus_replay', 'ewc_lambda_100', 'ewc_lambda_1000', 'ewc_lambda_5000', 'replay_ratio_5pct', 'replay_ratio_10pct', 'replay_ratio_20pct', 'few_shot']


In [4]:
# 4. Overall methods comparison: Avg F1 and AvgAcc

methods = [

    "static_joint",

    "naive_sequential",

    "ewc_only",

    "replay_only",

    "ewc_plus_replay",

]

labels = [

    "Static joint",

    "Naive sequential",

    "EWC only",

    "Replay only",

    "EWC + Replay",

]



avg_f1 = [all_results[m]["avg_f1"] for m in methods]

avg_acc = [all_results[m]["cl_metrics"]["AvgAcc"] for m in methods]



x = np.arange(len(methods))

width = 0.35



fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)



# Avg F1

axes[0].bar(x, avg_f1, width, color="steelblue", edgecolor="black")

axes[0].set_title("Average F1 by Method")

axes[0].set_xticks(x)

axes[0].set_xticklabels(labels, rotation=30, ha="right")

axes[0].set_ylabel("Average F1")

axes[0].set_ylim(0.0, 1.0)



# AvgAcc

axes[1].bar(x, avg_acc, width, color="darkorange", edgecolor="black")

axes[1].set_title("Average Accuracy by Method")

axes[1].set_xticks(x)

axes[1].set_xticklabels(labels, rotation=30, ha="right")

axes[1].set_ylim(0.0, 1.0)



fig.suptitle("Static vs. Continual Learning Methods")

fig.tight_layout(rect=[0, 0.02, 1, 0.95])



methods_fig_path = FIGURES_DIR / "methods_overview.png"

fig.savefig(methods_fig_path, dpi=200, bbox_inches="tight")

plt.close(fig)



print(f"Saved methods comparison figure to {methods_fig_path}")

Saved methods comparison figure to /Users/oluwadabiraomotoso/Desktop/CMPE 401/antidote/figures/methods_overview.png


In [5]:
# 5. Few-shot benefit: from_scratch vs cl_pretrained

few_shot = all_results["few_shot"]



shots = [5, 10]

from_scratch = [

    few_shot["few_shot_5"]["from_scratch"],

    few_shot["few_shot_10"]["from_scratch"],

]

cl_pretrained = [

    few_shot["few_shot_5"]["cl_pretrained"],

    few_shot["few_shot_10"]["cl_pretrained"],

]



x = np.arange(len(shots))

width = 0.35



fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(x - width / 2, from_scratch, width, label="From scratch", color="lightgray", edgecolor="black")

ax.bar(x + width / 2, cl_pretrained, width, label="CL-pretrained", color="seagreen", edgecolor="black")



ax.set_xticks(x)

ax.set_xticklabels(["5-shot", "10-shot"])

ax.set_ylabel("F1")

ax.set_ylim(0.0, 1.0)

ax.set_title("Few-Shot Detection: Benefit of CL-Pretrained Model")

ax.legend()



fig.tight_layout()



fewshot_fig_path = FIGURES_DIR / "fewshot_benefit.png"

fig.savefig(fewshot_fig_path, dpi=200, bbox_inches="tight")

plt.close(fig)



print(f"Saved few-shot comparison figure to {fewshot_fig_path}")

Saved few-shot comparison figure to /Users/oluwadabiraomotoso/Desktop/CMPE 401/antidote/figures/fewshot_benefit.png
